# 02 — Feature Engineering

## Objective

Convert the nanoparticle formulation design space into numerical features for UMAP analysis and downstream machine-learning models.

## Main tasks

1. Load and clean formulation data
2. Encode glucan, carrier, and formulation variables
3. Calculate SMILES-derived carrier descriptors
4. Add carrier family, PEG MW, polymer MW, and NH2 information
5. Merge Carrier 1 and Carrier 2 physicochemical properties into each formulation
6. Standardize continuous carrier descriptors
7. Generate full and property-based feature matrices
8. Export UMAP-ready feature tables


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Lipinski, rdMolDescriptors

In [2]:
# Project directories
PROJECT_ROOT = Path("..")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Input files
FORMULATION_FILE = RAW_DIR / "Dataset.csv"
CARRIER_FILE = RAW_DIR / "DrugCarriers Profile.csv"

# Output file
OUTPUT_FILE = PROCESSED_DIR / "formulation_features_chemical.csv"


In [3]:
df = pd.read_csv(FORMULATION_FILE)

# Remove unnecessary columns
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (6575, 16)


,ID,Glucan Type,Modification,Carrier 1,Carrier 2 (NH2),Carrier 1 Mass Ratio,Carrier 2 Mass Ratio,Glucan MW,Glucan / Carrier 2 Mass Ratio,Particle Size (nm),PDI,Zeta Potential (mV),Cell Uptake (%),Cell Uptake (MFI),M1 Marker (MFI),M2 Marker (MFI)
0,YG-SA-D1-N1-R90-10-MH-G01,Yeast Glucan,Succinic anhydride modification,D1,N1,0.9,0.1,High,0.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,YG-SA-D1-N1-R90-10-MH-G08,Yeast Glucan,Succinic anhydride modification,D1,N1,0.9,0.1,High,0.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,YG-SA-D1-N1-R90-10-MH-G20,Yeast Glucan,Succinic anhydride modification,D1,N1,0.9,0.1,High,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,YG-SA-D1-N1-R90-10-MM-G01,Yeast Glucan,Succinic anhydride modification,D1,N1,0.9,0.1,Medium,0.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,YG-SA-D1-N1-R90-10-MM-G08,Yeast Glucan,Succinic anhydride modification,D1,N1,0.9,0.1,Medium,0.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
categorical_features = [
    "Glucan Type",
    "Modification",
    "Carrier 1",
    "Carrier 2 (NH2)",
    "Glucan MW"
]

numeric_features = [
    "Carrier 1 Mass Ratio",
    "Carrier 2 Mass Ratio",
    "Glucan / Carrier 2 Mass Ratio"
]

features = categorical_features + numeric_features

X = df[features].copy()

X.head()

,Glucan Type,Modification,Carrier 1,Carrier 2 (NH2),Glucan MW,Carrier 1 Mass Ratio,Carrier 2 Mass Ratio,Glucan / Carrier 2 Mass Ratio
0,Yeast Glucan,Succinic anhydride modification,D1,N1,High,0.9,0.1,0.1
1,Yeast Glucan,Succinic anhydride modification,D1,N1,High,0.9,0.1,0.8
2,Yeast Glucan,Succinic anhydride modification,D1,N1,High,0.9,0.1,2.0
3,Yeast Glucan,Succinic anhydride modification,D1,N1,Medium,0.9,0.1,0.1
4,Yeast Glucan,Succinic anhydride modification,D1,N1,Medium,0.9,0.1,0.8


In [5]:
X[categorical_features] = (
    X[categorical_features]
    .fillna("None")
)
X[numeric_features] = (
    X[numeric_features]
    .fillna(0)
)
X.isnull().sum()

Glucan Type                      0
Modification                     0
Carrier 1                        0
Carrier 2 (NH2)                  0
Glucan MW                        0
Carrier 1 Mass Ratio             0
Carrier 2 Mass Ratio             0
Glucan / Carrier 2 Mass Ratio    0
dtype: int64

In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        (
            "num",
            StandardScaler(),
            numeric_features
        )
    ]
)
X_processed = preprocessor.fit_transform(X)

print("Original shape:", X.shape)
print("Processed shape:", X_processed.shape)


Original shape: (6575, 8)
Processed shape: (6575, 27)


In [7]:
feature_names = (
    preprocessor
    .get_feature_names_out()
)
X_processed = pd.DataFrame(
    X_processed,
    columns=feature_names,
    index=df.index
)

X_processed.head()

,cat__Glucan Type_Barley Glucan,cat__Glucan Type_Blank NP,cat__Glucan Type_Curdlan,cat__Glucan Type_Yeast Glucan,cat__Modification_H2O2 oxidation,cat__Modification_None,cat__Modification_Succinic anhydride modification,cat__Carrier 1_D1,cat__Carrier 1_D2,cat__Carrier 1_D3,...,cat__Carrier 2 (NH2)_N6,cat__Carrier 2 (NH2)_None,cat__Glucan MW_High,cat__Glucan MW_Low,cat__Glucan MW_Medium,cat__Glucan MW_None,cat__Glucan MW_Unspecified,num__Carrier 1 Mass Ratio,num__Carrier 2 Mass Ratio,num__Glucan / Carrier 2 Mass Ratio
0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.222246,-1.222246,-1.082958
1,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.222246,-1.222246,-0.193934
2,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.222246,-1.222246,1.330107
3,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.222246,-1.222246,-1.082958
4,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.222246,-1.222246,-0.193934


In [8]:
# Add formulation ID
processed_df = pd.concat(
    [
        df[["ID"]],
        X_processed
    ],
    axis=1
)

print("Final shape:", processed_df.shape)
processed_df.head()

Final shape: (6575, 28)


,ID,cat__Glucan Type_Barley Glucan,cat__Glucan Type_Blank NP,cat__Glucan Type_Curdlan,cat__Glucan Type_Yeast Glucan,cat__Modification_H2O2 oxidation,cat__Modification_None,cat__Modification_Succinic anhydride modification,cat__Carrier 1_D1,cat__Carrier 1_D2,...,cat__Carrier 2 (NH2)_N6,cat__Carrier 2 (NH2)_None,cat__Glucan MW_High,cat__Glucan MW_Low,cat__Glucan MW_Medium,cat__Glucan MW_None,cat__Glucan MW_Unspecified,num__Carrier 1 Mass Ratio,num__Carrier 2 Mass Ratio,num__Glucan / Carrier 2 Mass Ratio
0,YG-SA-D1-N1-R90-10-MH-G01,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.222246,-1.222246,-1.082958
1,YG-SA-D1-N1-R90-10-MH-G08,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.222246,-1.222246,-0.193934
2,YG-SA-D1-N1-R90-10-MH-G20,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.222246,-1.222246,1.330107
3,YG-SA-D1-N1-R90-10-MM-G01,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.222246,-1.222246,-1.082958
4,YG-SA-D1-N1-R90-10-MM-G08,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.222246,-1.222246,-0.193934


In [9]:
# Baseline formulation-only feature table (checkpoint)
print("Baseline feature table shape:", processed_df.shape)
print("Baseline missing values:", processed_df.isnull().sum().sum())


Baseline feature table shape: (6575, 28)
Baseline missing values: 0


In [10]:
carrier_df = pd.read_csv(CARRIER_FILE)

carrier_df = carrier_df.loc[
    :,
    ~carrier_df.columns.str.contains("^Unnamed")
]

# Remove leading/trailing spaces in column names
carrier_df.columns = carrier_df.columns.str.strip()

print("Carrier dataset shape:", carrier_df.shape)
print(carrier_df.columns.tolist())

carrier_df.head()


Carrier dataset shape: (11, 4)
['Carrier_ID', 'Chemical_Name', 'Carrier_Name', 'SMILES']


,Carrier_ID,Chemical_Name,Carrier_Name,SMILES
0,D1,"1,2-distearoyl-sn-glycero-3-phosphoethanolamin...",DSPE-mPEG,CCCCCCCCCCCCCCCCCC(=O)OCC(COP(=O)([O-])OCCNC(=...
1,D2,poly(lactic-co-glycolic acid)-methoxy poly(eth...,PLGA-mPEG,COCCOC(=O)COC(=O)C(C)O
2,D3,poly(lactide)-methoxy poly(ethylene glycol),PLA-mPEG,COCCOC(=O)C(C)O
3,D4,poly(ε-caprolactone)-methoxy poly(ethylene gly...,PCL-mPEG,COCCOC(=O)CCCCCO
4,D5,Poloxamer 407,F-127,OCCOCC(C)OCCO


In [11]:
carrier_df["Mol"] = carrier_df["SMILES"].apply(
    lambda x: Chem.MolFromSmiles(x)
    if pd.notna(x)
    else None
)

print(
    "Valid SMILES:",
    carrier_df["Mol"].notna().sum(),
    "/",
    len(carrier_df)
)

Valid SMILES: 11 / 11


In [12]:
def safe_descriptor(mol, func):
    return func(mol) if mol is not None else np.nan

carrier_df["SMILES_MolWt"] = carrier_df["Mol"].apply(
    lambda x: safe_descriptor(x, Descriptors.MolWt)
)
carrier_df["LogP"] = carrier_df["Mol"].apply(
    lambda x: safe_descriptor(x, Crippen.MolLogP)
)
carrier_df["TPSA"] = carrier_df["Mol"].apply(
    lambda x: safe_descriptor(x, rdMolDescriptors.CalcTPSA)
)
carrier_df["HBD"] = carrier_df["Mol"].apply(
    lambda x: safe_descriptor(x, Lipinski.NumHDonors)
)
carrier_df["HBA"] = carrier_df["Mol"].apply(
    lambda x: safe_descriptor(x, Lipinski.NumHAcceptors)
)
carrier_df["RotatableBonds"] = carrier_df["Mol"].apply(
    lambda x: safe_descriptor(x, Lipinski.NumRotatableBonds)
)


In [13]:
display_cols = [
    "Carrier_ID", "SMILES_MolWt", "LogP", "TPSA",
    "HBD", "HBA", "RotatableBonds"
]
carrier_df[display_cols]


,Carrier_ID,SMILES_MolWt,LogP,TPSA,HBD,HBA,RotatableBonds
0,D1,872.151,8.8426,158.75,1,11,45
1,D2,206.194,-0.9000,82.06,1,6,6
2,D3,148.158,-0.4432,55.76,1,4,4
3,D4,190.239,0.7287,55.76,1,4,8
4,D5,164.201,-0.6073,58.92,2,4,7
5,N1,896.242,11.9615,216.94,4,12,47
6,N2,191.183,-1.5877,98.85,2,6,5
7,N3,133.147,-1.1309,72.55,2,4,3
8,N4,175.228,0.0410,72.55,2,4,7
9,N5,163.217,-0.6409,64.71,2,4,7


## Carrier material properties

Add physically meaningful carrier information in addition to the SMILES-derived descriptors.

`SMILES_MolWt` is an RDKit descriptor of the SMILES representation and is **not** the actual molecular weight of the purchased polymer. Actual PEG/polymer molecular weights are therefore stored separately.

The values below should be checked against the final supplier specifications before final model training.


In [14]:
material_info = pd.DataFrame({
    "Carrier_ID": [
        "D1", "D2", "D3", "D4", "D5",
        "N1", "N2", "N3", "N4", "N5", "N6"
    ],

    # Broad material family
    "Carrier_Family": [
        "Lipid", "Lipid", "PLGA", "PLA", "PCL",
        "Lipid", "Lipid", "PLGA", "PLA", "PCL", "Poloxamer"
    ],

    # PEG molecular weight (Da)
    "PEG_MW": [
        2000, 2000, 2000, 2000, 2000,
        2000, 2000, 2000, 2000, 0, 2000
    ],

    # Nominal hydrophobic/polymer block molecular weight (Da)
    "Polymer_MW": [
        748, 10000, 10000, 10000, 10000,
        748, 10000, 10000, 10000, 12600, 748
    ],

    # Amine functionality
    "NH2": [
        0, 0, 0, 0, 0,
        1, 1, 1, 1, 1, 1
    ]
})

material_info


,Carrier_ID,Carrier_Family,PEG_MW,Polymer_MW,NH2
0,D1,Lipid,2000,748,0
1,D2,Lipid,2000,10000,0
2,D3,PLGA,2000,10000,0
3,D4,PLA,2000,10000,0
4,D5,PCL,2000,10000,0
5,N1,Lipid,2000,748,1
6,N2,Lipid,2000,10000,1
7,N3,PLGA,2000,10000,1
8,N4,PLA,2000,10000,1
9,N5,PCL,0,12600,1


In [15]:
# Merge material properties into the carrier profile
carrier_df = carrier_df.merge(
    material_info,
    on="Carrier_ID",
    how="left"
)

carrier_df[[
    "Carrier_ID",
    "Carrier_Name",
    "Carrier_Family",
    "PEG_MW",
    "Polymer_MW",
    "NH2",
    "SMILES_MolWt",
    "LogP",
    "TPSA",
    "HBD",
    "HBA",
    "RotatableBonds"
]]


,Carrier_ID,Carrier_Name,Carrier_Family,PEG_MW,Polymer_MW,NH2,SMILES_MolWt,LogP,TPSA,HBD,HBA,RotatableBonds
0,D1,DSPE-mPEG,Lipid,2000,748,0,872.151,8.8426,158.75,1,11,45
1,D2,PLGA-mPEG,Lipid,2000,10000,0,206.194,-0.9000,82.06,1,6,6
2,D3,PLA-mPEG,PLGA,2000,10000,0,148.158,-0.4432,55.76,1,4,4
3,D4,PCL-mPEG,PLA,2000,10000,0,190.239,0.7287,55.76,1,4,8
4,D5,F-127,PCL,2000,10000,0,164.201,-0.6073,58.92,2,4,7
5,N1,DSPE-PEG-NH2,Lipid,2000,748,1,896.242,11.9615,216.94,4,12,47
6,N2,PLGA-PEG-NH2,Lipid,2000,10000,1,191.183,-1.5877,98.85,2,6,5
7,N3,PLA-PEG-NH2,PLGA,2000,10000,1,133.147,-1.1309,72.55,2,4,3
8,N4,PCL-PEG-NH2,PLA,2000,10000,1,175.228,0.0410,72.55,2,4,7
9,N5,F-127-NH2,PCL,0,12600,1,163.217,-0.6409,64.71,2,4,7


## Encode carrier family

Carrier family is retained as an explicit material feature so PLGA, PLA, PCL, lipid, and poloxamer carriers are not represented only through generic molecular descriptors.


In [16]:
family_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

family_encoded = family_encoder.fit_transform(
    carrier_df[["Carrier_Family"]].fillna("Unknown")
)

family_names = family_encoder.get_feature_names_out(
    ["Carrier_Family"]
)

family_df = pd.DataFrame(
    family_encoded,
    columns=family_names,
    index=carrier_df.index
)

carrier_df = pd.concat(
    [carrier_df, family_df],
    axis=1
)

family_df.head()


,Carrier_Family_Lipid,Carrier_Family_PCL,Carrier_Family_PLA,Carrier_Family_PLGA,Carrier_Family_Poloxamer
0,1.0,0.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,1.0,0.0
3,0.0,0.0,1.0,0.0,0.0
4,0.0,1.0,0.0,0.0,0.0


## Build Carrier 1 and Carrier 2 property tables

The same carrier-property profile is used for both formulation positions. Prefixes `C1_` and `C2_` preserve which carrier position each property belongs to.


In [17]:
descriptor_cols = [
    "PEG_MW",
    "Polymer_MW",
    "NH2",
    "SMILES_MolWt",
    "LogP",
    "TPSA",
    "HBD",
    "HBA",
    "RotatableBonds"
]

family_cols = [
    col for col in carrier_df.columns
    if col.startswith("Carrier_Family_")
]

carrier_feature_cols = descriptor_cols + family_cols

print("Carrier numerical/material features:")
print(carrier_feature_cols)


Carrier numerical/material features:
['PEG_MW', 'Polymer_MW', 'NH2', 'SMILES_MolWt', 'LogP', 'TPSA', 'HBD', 'HBA', 'RotatableBonds', 'Carrier_Family_Lipid', 'Carrier_Family_PCL', 'Carrier_Family_PLA', 'Carrier_Family_PLGA', 'Carrier_Family_Poloxamer']


In [18]:
carrier1 = carrier_df[
    ["Carrier_ID"] + carrier_feature_cols
].copy()

carrier1 = carrier1.rename(
    columns={
        "Carrier_ID": "Carrier 1",
        **{
            col: f"C1_{col}"
            for col in carrier_feature_cols
        }
    }
)

carrier2 = carrier_df[
    ["Carrier_ID"] + carrier_feature_cols
].copy()

carrier2 = carrier2.rename(
    columns={
        "Carrier_ID": "Carrier 2 (NH2)",
        **{
            col: f"C2_{col}"
            for col in carrier_feature_cols
        }
    }
)

print("Carrier 1 property table:", carrier1.shape)
print("Carrier 2 property table:", carrier2.shape)


Carrier 1 property table: (11, 15)
Carrier 2 property table: (11, 15)


## Merge carrier properties into formulation dataset


In [19]:
df_features = df.merge(
    carrier1,
    on="Carrier 1",
    how="left"
)

df_features = df_features.merge(
    carrier2,
    on="Carrier 2 (NH2)",
    how="left"
)

print("Merged formulation dataset:", df_features.shape)


Merged formulation dataset: (6575, 44)


## Validate carrier matching

A named carrier should never have all of its carrier-property features missing. Blank Carrier 2 entries are allowed.


In [20]:
c1_feature_cols = [
    f"C1_{col}" for col in carrier_feature_cols
]

c2_feature_cols = [
    f"C2_{col}" for col in carrier_feature_cols
]

unmatched_c1 = df_features.loc[
    df_features["Carrier 1"].notna()
    & df_features[c1_feature_cols].isna().all(axis=1),
    "Carrier 1"
].unique()

unmatched_c2 = df_features.loc[
    df_features["Carrier 2 (NH2)"].notna()
    & df_features[c2_feature_cols].isna().all(axis=1),
    "Carrier 2 (NH2)"
].unique()

print("Unmatched Carrier 1:", unmatched_c1)
print("Unmatched Carrier 2:", unmatched_c2)


Unmatched Carrier 1: []
Unmatched Carrier 2: []


## Standardize continuous carrier descriptors

Continuous carrier descriptors are standardized. Binary family and NH2 variables are kept as 0/1 so their interpretation remains explicit.


In [21]:
continuous_descriptor_cols = [
    "PEG_MW",
    "Polymer_MW",
    "SMILES_MolWt",
    "LogP",
    "TPSA",
    "HBD",
    "HBA",
    "RotatableBonds"
]

continuous_features = (
    [f"C1_{col}" for col in continuous_descriptor_cols]
    +
    [f"C2_{col}" for col in continuous_descriptor_cols]
)

binary_features = (
    ["C1_NH2", "C2_NH2"]
    +
    [f"C1_{col}" for col in family_cols]
    +
    [f"C2_{col}" for col in family_cols]
)

carrier_continuous = (
    df_features[continuous_features]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
)

carrier_scaler = StandardScaler()

carrier_continuous_scaled = pd.DataFrame(
    carrier_scaler.fit_transform(carrier_continuous),
    columns=continuous_features,
    index=df.index
)

carrier_binary = (
    df_features[binary_features]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
)

carrier_features_processed = pd.concat(
    [
        carrier_continuous_scaled,
        carrier_binary
    ],
    axis=1
)

print("Carrier property feature shape:",
      carrier_features_processed.shape)


Carrier property feature shape: (6575, 28)


## Build two final feature sets

Two feature tables are exported:

- **Full feature set** — formulation one-hot variables + carrier identity + chemical/material properties.
- **Property-based feature set** — removes Carrier 1/Carrier 2 identity one-hot columns while retaining carrier family, SMILES descriptors, MW, NH2, glucan, and formulation-ratio information.

The property-based table is recommended for the UMAP that should reflect material similarity rather than carrier labels.


In [22]:
X_full = pd.concat(
    [
        X_processed,
        carrier_features_processed
    ],
    axis=1
)

carrier_identity_cols = [
    col for col in X_full.columns
    if col.startswith("cat__Carrier 1_")
    or col.startswith("cat__Carrier 2 (NH2)_")
]

X_property = X_full.drop(
    columns=carrier_identity_cols
)

print("Baseline formulation features:", X_processed.shape[1])
print("Carrier property features:",
      carrier_features_processed.shape[1])
print("Full feature count:", X_full.shape[1])
print("Property-based feature count:", X_property.shape[1])
print("Removed carrier identity features:",
      len(carrier_identity_cols))


Baseline formulation features: 27
Carrier property features: 28
Full feature count: 55
Property-based feature count: 43
Removed carrier identity features: 12


## Export processed feature tables


In [23]:
FULL_OUTPUT_FILE = (
    PROCESSED_DIR
    / "formulation_features_chemical_full.csv"
)

PROPERTY_OUTPUT_FILE = (
    PROCESSED_DIR
    / "formulation_features_property_based.csv"
)

full_df = pd.concat(
    [df[["ID"]], X_full],
    axis=1
)

property_df = pd.concat(
    [df[["ID"]], X_property],
    axis=1
)

full_df.to_csv(
    FULL_OUTPUT_FILE,
    index=False
)

property_df.to_csv(
    PROPERTY_OUTPUT_FILE,
    index=False
)

print("Full feature table:")
print(FULL_OUTPUT_FILE, full_df.shape)

print("\nProperty-based feature table:")
print(PROPERTY_OUTPUT_FILE, property_df.shape)


Full feature table:
..\data\processed\formulation_features_chemical_full.csv (6575, 56)

Property-based feature table:
..\data\processed\formulation_features_property_based.csv (6575, 44)


## Final quality check


In [24]:
print("=" * 60)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 60)

print("Formulations:", len(df))
print("Carrier profiles:", len(carrier_df))
print(
    "Valid SMILES:",
    carrier_df["Mol"].notna().sum(),
    "/",
    len(carrier_df)
)

print("\nFull feature count:",
      full_df.shape[1] - 1)

print("Property-based feature count:",
      property_df.shape[1] - 1)

print("\nFull table missing values:",
      full_df.isnull().sum().sum())

print("Property table missing values:",
      property_df.isnull().sum().sum())

print("Duplicate formulation IDs:",
      df["ID"].duplicated().sum())

print("\nUse for property-based UMAP:")
print(PROPERTY_OUTPUT_FILE)


FEATURE ENGINEERING SUMMARY
Formulations: 6575
Carrier profiles: 11
Valid SMILES: 11 / 11

Full feature count: 55
Property-based feature count: 43

Full table missing values: 0
Property table missing values: 0
Duplicate formulation IDs: 0

Use for property-based UMAP:
..\data\processed\formulation_features_property_based.csv
